This code reproduces the buy-the-dip strategy [described here](https://www.crackingmarkets.com/buy-the-dip/). The idea is to buy stocks that are trading above their long-term average but have done poorly over the last few days, then sell them shortly after. The default parameters are a 200-day moving average as the trend filter, a 5-day RSI below 20 as the dip signal, and a holding period of 5 trading days.

The data are daily adjusted closing prices for 40 stocks listed on the Warsaw Stock Exchange, downloaded from Yahoo Finance with trades taken from 2006. Each trade’s return is reduced by an assumed transaction cost of 0.78%, which covers both the buy and the sell.


In [14]:
import time, urllib.request
from pathlib import Path
import pandas as pd
TICKERS = ["ing", "acp", "bft", "mil", "tpe", "xtb", "car", "ena", "dvl", "cps",
           "dia", "bhw", "bnp", "dom", "rbw", "ase", "nwg", "gpp", "gpw", "apr",
           "wpl", "jsw", "abe", "cbf", "neu", "snt", "vrc", "vox", "pep", "mrb",
           "att", "asb", "txt", "mbr", "eat", "lbw", "hug", "eur", "11b", "ten"]

PRICES = Path("data/prices")
SMA_LEN, RSI_LEN, RSI_MAX, HOLD = 200, 5, 20, 5
BET = 1000
COST = 0.0078
START = "2006-01-01"

In [7]:
import yfinance as yf

raw = yf.download([t.upper() + ".WA" for t in TICKERS], start="2005-01-01", auto_adjust=True)["Close"]
close = raw.rename(columns=lambda c: c.replace(".WA", "").lower()).dropna(how="all")
assert not close.empty, "no prices loaded"
print(close.shape, close.index[0].date(), close.index[-1].date())

/Users/dzmitryshautsou/PycharmProjects/PythonProject3/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[**********************95%*********************  ]  38 of 40 completedFailed to get ticker 'EAT.WA' reason: Failed to perform, curl: (28) Operation timed out after 10001 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  40 of 40 completed

(6906, 40) 2000-01-03 2026-11-10


In [8]:
sma = close.rolling(SMA_LEN).mean()
change = close.diff()
gain = change.clip(lower=0).ewm(alpha=1 / RSI_LEN, adjust=False).mean()
loss = (-change).clip(lower=0).ewm(alpha=1 / RSI_LEN, adjust=False).mean()
rsi = 100 - 100 / (1 + gain / loss)

In [10]:
signal = (close > sma) & (rsi < RSI_MAX) & (close.index >= START)[:, None]
exit_price = close.ffill().shift(-HOLD)
exit_date = close.index.to_series().shift(-HOLD)

trades = []
for t in close.columns:
    free_from = close.index[0]
    for day in signal.index[signal[t]]:
        if day < free_from or pd.isna(exit_date[day]):
            continue
        trades.append({"ticker": t, "buy_date": day, "sell_date": exit_date[day],
                       "buy": close.at[day, t], "sell": exit_price.at[day, t]})
        free_from = exit_date[day]
trades = pd.DataFrame(trades).sort_values("buy_date")
trades["ret"] = trades["sell"] / trades["buy"] - 1 - COST
trades["pnl"] = trades["ret"] * BET

In [11]:
print(f"trades:     {len(trades)}")
print(f"win rate:   {(trades.ret > 0).mean():.1%}")
print(f"avg return: {trades.ret.mean():.2%} per trade, after costs")
print(f"total P&L:  {trades.pnl.sum():,.0f} zl")
print(trades.groupby(trades.buy_date.dt.year)["ret"].agg(["count", "mean"]))
trades.to_csv("trades.csv", index=False)

trades:     1381
win rate:   52.9%
avg return: 0.58% per trade, after costs
total P&L:  8,043 zl
          count      mean
buy_date                 
2006         42 -0.007915
2007         32  0.002007
2008          3 -0.111804
2009         39  0.018981
2010         52  0.007562
2011         47 -0.005124
2012         58  0.002225
2013        119  0.004406
2014         68 -0.007879
2015         74  0.014097
2016         65  0.003790
2017        104  0.011643
2018         60  0.003348
2019         85 -0.004493
2020         80  0.013484
2021        104  0.007426
2022         33  0.010591
2023         82  0.011094
2024         72  0.016262
2025         97  0.015029
2026         65 -0.006494
